# Creating a Pseudonymized PII Lookup Tables

In this lesson we'll create a pseudonymized key for storing potentially sensitive user data.  
Our approach in this notebook is fairly straightforward; some industries may require more elaborate de-identification to guarantee privacy.

We'll examine design patterns for ensuring PII is stored securely and updated accurately. 

##### Objectives
- Describe the purpose of "salting" before hashing
- Apply salted hashing to sensitive data(user_id)
- Apply tokenization to sensitive data(user_id)

##### Creates the Following
  1. **customers.csv** is coming vrom databricks_simulated_retail_customer_data.

  1. Hashing: Handled in table **customer_lookup_hashed**

  1. Tokenization: Handled in tables **registered_re_tokens** and **user_lookup_tokenized**


### C1. with salted hashing

Create a function to register this logic to the current database under the name **`salted_hash`**. This will allow this logic to be called by any user with appropriate permissions on this function. 

Note that it is theoretically possible to link the original key and pseudo-ID if the hash function and the salt are known. Here, we use this method to add a layer of obfuscation; in production, you may wish to have a much more sophisticated hashing method.

In [0]:
%sql
use catalog dbx_catalog;
use schema dbx_schema;

In [0]:
salt = "BEANS"
import pyspark.sql.functions as F  
# Define function to pseudonymize with salted hashing    
def salted_hash(id):
    return F.sha2(F.concat(id, F.lit(salt)), 256)

In [0]:
df =spark.read.csv('/Volumes/databricks_simulated_retail_customer_data/v01/source_files/customers.csv',header=True,inferSchema=True)

In [0]:
display(df.show(5))

+-----------+------+--------+--------------------+-----+-------------------+--------+------------+----------+----+-------+--------+------------------+-----------------+--------------------+----------+----------+---------------+---------------+
|customer_id|tax_id|tax_code|       customer_name|state|               city|postcode|      street|    number|unit| region|district|               lon|              lat|     ship_to_address|valid_from|  valid_to|units_purchased|loyalty_segment|
+-----------+------+--------+--------------------+-----+-------------------+--------+------------+----------+----+-------+--------+------------------+-----------------+--------------------+----------+----------+---------------+---------------+
|   11123757|  NULL|    NULL|     SMITH,  SHIRLEY|   IN|             BREMEN| 46506.0| N CENTER ST|     521.0|NULL|Indiana|    50.0|       -86.1465825|       41.4507625|IN, 46506.0, N CE...|1532824233|1548137353|             34|              3|
|   30585978|  NULL|    

In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- valid_from: integer (nullable = true)
 |-- valid_to: string (nullable = true)
 |-- units_purchased: integer (nullable = true)
 |-- loyalty_segment: integer (nullable = true)



In [0]:
df.select(salted_hash(F.col("customer_id")).alias("customer_alt_id"),
                  "customer_id", "postcode", "customer_name").show(5)

+--------------------+-----------+--------+--------------------+
|     customer_alt_id|customer_id|postcode|       customer_name|
+--------------------+-----------+--------+--------------------+
|3e6ce1a4bbacc5bc7...|   11123757| 46506.0|     SMITH,  SHIRLEY|
|c7bb4fd53e1e8feab...|   30585978|       0|STEPHENS,  GERALD...|
|72a6838f6ad2f1949...|     349822|   22181|     GUZMAN,  CARMEN|
|121c177984f7cc123...|   27652636| 53058.0| HASSETT,  PATRICK J|
|3929ab9d13b383a48...|   14437343| 43228.0|     HENTZ,  DIANA L|
+--------------------+-----------+--------+--------------------+
only showing top 5 rows


In [0]:
df.select(salted_hash(F.col("customer_id")).alias("customer_alt_id"),
                  "customer_id", "postcode", "customer_name").write.saveAsTable("dbx_catalog.dbx_schema.customerid_hashed")

In [0]:
df.write.mode("overwrite").saveAsTable("dbx_catalog.dbx_schema.customer")

# Tokenized table

In [0]:
df.select("customer_id").distinct().withColumn('token',F.expr("uuid()")).write.saveAsTable("dbx_catalog.dbx_schema.customer_lookup_table")

In [0]:
spark.sql("describe  dbx_catalog.dbx_schema.customer_lookup_table").display()

col_name,data_type,comment
customer_id,int,null
token,string,null


In [0]:
spark.sql("describe  dbx_catalog.dbx_schema.customer").display()

col_name,data_type,comment
customer_id,int,null
tax_id,string,null
tax_code,string,null
customer_name,string,null
state,string,null
city,string,null
postcode,string,null
street,string,null
number,string,null
unit,string,null


In [0]:
spark.table ("dbx_catalog.dbx_schema.customer")\
     .join(spark.table("dbx_catalog.dbx_schema.customer_lookup_table"), "customer_id", "left")\
     .drop("customer_id")\
     .withColumnRenamed("token", "alt_id")\
     .write\
     .mode("overwrite")\
     .option("overwriteSchema", "true")\
     .saveAsTable("dbx_catalog.dbx_schema.customer")


In [0]:
%sql
select * from dbx_catalog.dbx_schema.customer limit 5

tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment,alt_id
NULL,NULL,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,NULL,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353,34,3,0d320f84-7fe9-4699-a3a0-641fb08a2adf
NULL,NULL,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,NULL,NULL,NULL,NULL,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,NULL,18,3,42adb6ac-bc30-424e-bece-feab5acbbe3a
NULL,NULL,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,NULL,VA,NULL,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,NULL,5,0,48c40434-8886-4729-80d2-cc4dbaaad793
NULL,NULL,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,NULL,NULL,NULL,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195,7,1,ff40f3bc-4d52-4dfd-8f77-0960cf2cc0c0
NULL,NULL,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,NULL,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,NULL,0,0,8910792b-c425-495c-b78e-fc04e7f9b3a2
